# Tutorial 08 — Propulsion and Endurance

Browse the hardware catalog, set up a propulsion system, analyze it at different
throttle settings, estimate endurance, and integrate with mission analysis.

**What you'll learn:**
- Browse motors, batteries, and propellers in the catalog
- Build a `PropulsionSystem` and run `propulsion.analyze()`
- Throttle sweep: thrust and efficiency curves
- Compute and compare endurance for different hardware combos
- Feed propulsion results into the mission discipline

In [1]:
import aerisplane as ap
from aerisplane.catalog import list_motors, list_batteries, list_propellers

motors = list_motors()
print(f"{'Name':<42} {'KV':>6} {'I_max':>7} {'mass':>7}")
print("-" * 67)
for m in sorted(motors, key=lambda x: x.kv):
    print(f"{m.name:<42} {m.kv:>6.0f} {m.max_current:>6.0f}A {m.mass*1000:>6.0f}g")

Name                                           KV   I_max    mass
-------------------------------------------------------------------
T-Motor MN4014 330KV                          330     22A    176g
T-Motor MN5212 340KV                          340     30A    215g
RCTimer 5010 360KV                            360     40A    190g
T-Motor MN3110 700KV                          700     16A    102g
T-Motor MN3110 780KV                          780     16A    102g
Emax MT2216 810KV                             810     20A     75g
Scorpion HKII-2221 900KV                      900     22A     68g
Emax MT2213 935KV                             935     20A     57g
T-Motor MN2213 950KV                          950     14A     60g
SunnySky X2212 980KV                          980     20A     52g
Hacker A20-26L                               1020     16A     72g
AXi 2217/20                                  1050     18A     95g
Dualsky ECO 2315C 1100KV                     1100     18A     65g
SunnySky

In [2]:
batteries = list_batteries()
print(f"{'Name':<42} {'Cap [Ah]':>9} {'V':>6} {'C':>5} {'mass':>7}")
print("-" * 74)
for b in sorted(batteries, key=lambda x: x.capacity_ah):
    print(f"{b.name:<42} {b.capacity_ah:>9.1f} {b.nominal_voltage:>6.1f} {b.c_rating:>5.0f}C {b.mass*1000:>6.0f}g")

Name                                        Cap [Ah]      V     C    mass
--------------------------------------------------------------------------
Tattu 4S 1800mAh 75C                             1.8   14.8    75C    218g
Gens Ace 3S 2200mAh 25C                          2.2   11.1    25C    162g
Turnigy Nano-tech 3S 2200mAh 25-50C              2.2   11.1    25C    156g
Ovonic 4S 2200mAh 50C                            2.2   14.8    50C    200g
Tattu 3S 2300mAh 45C                             2.3   11.1    45C    178g
Tattu 4S 3300mAh 45C                             3.3   14.8    45C    302g
Turnigy Nano-tech 6S 3300mAh 45-90C              3.3   22.2    45C    480g
Ovonic 6S 3300mAh 50C                            3.3   22.2    50C    480g
Gens Ace 4S 4000mAh 45C                          4.0   14.8    45C    390g
Turnigy Nano-tech 4S 5000mAh 25-50C              5.0   14.8    25C    480g
Tattu 4S 5200mAh 45C                             5.2   14.8    45C    470g
Gens Ace 6S 6000mAh 30C   

In [3]:
import matplotlib
matplotlib.use('Agg')  # non-interactive backend for notebook execution
import matplotlib.pyplot as plt
import numpy as np

from aerisplane.catalog.motors import sunnysky_x2216_1250
from aerisplane.catalog.batteries import tattu_4s_5200
from aerisplane.catalog.propellers import apc_10x4_7sf
from aerisplane.core.propulsion import ESC, PropulsionSystem
from aerisplane.propulsion import analyze as prop_analyze

propulsion = PropulsionSystem(
    motor=sunnysky_x2216_1250,
    propeller=apc_10x4_7sf,
    battery=tattu_4s_5200,
    esc=ESC(name="generic_60A", max_current=60.0, mass=0.025),
)

# Build a simple aircraft for the analysis
wing = ap.Wing(
    name="main_wing",
    symmetric=True,
    xsecs=[
        ap.WingXSec(xyz_le=[0.00, 0.00, 0.00], chord=0.28, airfoil=ap.Airfoil(name="ag35")),
        ap.WingXSec(xyz_le=[0.07, 1.20, 0.05], chord=0.14, twist=-2.0,
                    airfoil=ap.Airfoil(name="ag35")),
    ],
)
aircraft = ap.Aircraft(name="MyPlane", wings=[wing], propulsion=propulsion)

# Single operating point — cruise throttle
cond = ap.FlightCondition(velocity=16.0, altitude=0.0)
result = prop_analyze(aircraft, cond, throttle=0.6)
print(result.report())

Propulsion Analysis (throttle=60%, V=16.0 m/s)
  Thrust               : 0.00 N
  Current              : 14.40 A
  RPM                  : 8994
  Motor efficiency     : 77.6%
  Propulsive efficiency: 0.0%
  Electrical power     : 127.9 W
  Shaft power          : 100.0 W
  C-rate               : 2.8 C
  Battery endurance    : 36.1 min



## Throttle sweep

How does thrust and efficiency change from idle to full throttle?

In [4]:
throttles = np.linspace(0.2, 1.0, 17)
thrusts, motor_effs, prop_effs, currents = [], [], [], []

for t in throttles:
    r = prop_analyze(aircraft, cond, throttle=t)
    thrusts.append(r.thrust_n)
    motor_effs.append(r.motor_efficiency * 100)
    prop_effs.append(r.propulsive_efficiency * 100)
    currents.append(r.current_a)

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].plot(throttles * 100, thrusts, "b-o", ms=4)
axes[0].set_xlabel("Throttle [%]")
axes[0].set_ylabel("Thrust [N]")
axes[0].set_title("Thrust")
axes[0].grid(True, alpha=0.3)

axes[1].plot(throttles * 100, motor_effs, "r-o", ms=4, label="Motor η")
axes[1].plot(throttles * 100, prop_effs,  "g-o", ms=4, label="Propulsive η")
axes[1].set_xlabel("Throttle [%]")
axes[1].set_ylabel("Efficiency [%]")
axes[1].set_title("Efficiency")
axes[1].legend()
axes[1].grid(True, alpha=0.3)

axes[2].plot(throttles * 100, currents, "m-o", ms=4)
axes[2].axhline(sunnysky_x2216_1250.max_current, color="r", ls="--", label="I_max")
axes[2].set_xlabel("Throttle [%]")
axes[2].set_ylabel("Current [A]")
axes[2].set_title("Battery current")
axes[2].legend()
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

/tmp/ipykernel_23388/3967087111.py:36: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Endurance comparison

Compare endurance for different battery choices at 60% cruise throttle.

In [5]:
from aerisplane.catalog.batteries import tattu_4s_3300, tattu_4s_5200, gens_ace_4s_4000, turnigy_nano_tech_4s_5000

battery_options = [
    ("Tattu 4S 3300", tattu_4s_3300),
    ("Tattu 4S 5200", tattu_4s_5200),
    ("Gens Ace 4S 4000", gens_ace_4s_4000),
    ("Nano-Tech 4S 5000", turnigy_nano_tech_4s_5000),
]

endurances, labels, masses = [], [], []

for label, batt in battery_options:
    prop_sys = PropulsionSystem(
        motor=sunnysky_x2216_1250,
        propeller=apc_10x4_7sf,
        battery=batt,
        esc=ESC(name="generic_60A", max_current=60.0, mass=0.025),
    )
    ac = ap.Aircraft(name="test", wings=[wing], propulsion=prop_sys)
    r = prop_analyze(ac, cond, throttle=0.6)
    endurances.append(r.battery_endurance_s / 60)
    labels.append(label)
    masses.append(batt.mass * 1000)

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.bar(labels, endurances, color="steelblue")
ax.bar_label(bars, fmt="%.1f min", padding=3)
ax2 = ax.twinx()
ax2.plot(labels, masses, "ro--", label="Battery mass [g]")
ax2.set_ylabel("Battery mass [g]", color="red")
ax.set_ylabel("Endurance [min]")
ax.set_title("Endurance by battery (60% throttle, V=16 m/s)")
plt.tight_layout()
plt.show()

/tmp/ipykernel_23388/3594418726.py:34: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Integration with mission analysis

The propulsion model feeds into `mission.analyze()` for accurate range and
endurance estimates accounting for different flight segments.

In [6]:
from aerisplane.weights import analyze as weight_analyze
from aerisplane.mission import analyze as mission_analyze
from aerisplane.mission.segments import Mission, Climb, Cruise, Loiter

wr = weight_analyze(aircraft)

mission = Mission(
    start_altitude=0.0,
    segments=[
        Climb(name="climb",  velocity=14.0, climb_rate=2.0, to_altitude=100.0),
        Cruise(name="cruise", velocity=18.0, altitude=100.0, distance=8000.0),
        Loiter(name="loiter", velocity=16.0, altitude=100.0, duration=600.0),
        Cruise(name="return", velocity=18.0, altitude=100.0, distance=8000.0),
    ]
)

mr = mission_analyze(aircraft, wr, mission)
print(mr.report())

AerisPlane Mission Analysis

Segment          Duration  Distance    Energy  Avg Power        Alt
---------------------------------------------------------------------------
climb                 50s      700m      2.6Wh     190.6W    0->100m
cruise               444s     8000m     27.8Wh     225.0W  100->100m
loiter               600s     9600m     25.2Wh     151.1W  100->100m
return               444s     8000m     27.8Wh     225.0W  100->100m
---------------------------------------------------------------------------
TOTAL               1539s    26300m     83.4Wh

Battery energy:   77.0 Wh
Energy used:      83.4 Wh
Energy margin:    0.0%
Mission time:     25.6 min
Status:           NOT FEASIBLE
